# E009 — Score-level fusion

We combine calibrated OOF scores from:
- **E007** image (PCA+LogReg+aug): calibrated log-odds
- **E008** audio (UBM+MAP+aug): calibrated log-odds

Three fusion strategies tested:
1. **Average** — equal weight, the naive baseline
2. **Grid search** — find optimal `w` in `score = w·audio + (1−w)·image`
3. **LogReg** — logistic regression learns weights from data

**Methodological note:** fusion is fit and evaluated on the same OOF pool.
This is slightly optimistic for LogReg (2 parameters fit on 222 samples),
but the alternative (nested 3-fold) would leave only 1 fold for training —
too unstable. We document this and treat OOF fusion numbers as upper bounds.

In [ ]:
from pathlib import Path
import sys, pickle
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from scipy.stats import norm as scipy_norm

from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "audio":     "#E67E22",
    "image":     "#8E44AD",
    "fusion":    "#E74C3C",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# Load calibrated OOF scores
with open(Path("../cache/calibration.pkl"), "rb") as f:
    calib = pickle.load(f)

oof_audio = calib["oof_audio_cal"]
oof_image = calib["oof_image_cal"]
y_all     = calib["y_all"]

print(f"Loaded OOF scores: {len(y_all)} samples, {y_all.sum()} target")
print(f"Audio cal range: [{oof_audio.min():.2f}, {oof_audio.max():.2f}]")
print(f"Image cal range: [{oof_image.min():.2f}, {oof_image.max():.2f}]")

## 1. Score space visualization

Where do target and non-target samples sit in the 2D (audio, image) score space?
This reveals whether the two modalities are correlated or complementary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 2D scatter
ax = axes[0]
ax.scatter(oof_audio[y_all==0], oof_image[y_all==0],
           c=COLORS["nontarget"], alpha=0.4, s=20, label="non-target")
ax.scatter(oof_audio[y_all==1], oof_image[y_all==1],
           c=COLORS["target"], alpha=0.9, s=80, label="target (m431)", zorder=5)
ax.axvline(0, color=COLORS["gray"], ls="--", lw=1, alpha=0.7, label="audio thresh=0")
ax.axhline(0, color=COLORS["gray"], ls=":",  lw=1, alpha=0.7, label="image thresh=0")
ax.set_xlabel("Audio calibrated score")
ax.set_ylabel("Image calibrated score")
ax.set_title("Score space: audio vs image (calibrated OOF)\nBoth axes: higher = more confident target")
ax.legend(fontsize=9)

# Marginal distributions overlay
ax = axes[1]
for scores, name, color, ls in [
    (oof_audio, "Audio E008", COLORS["audio"], "-"),
    (oof_image, "Image E007", COLORS["image"], "--"),
]:
    eer_c, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    bins = np.linspace(scores.min(), scores.max(), 40)
    ax.hist(scores[y_all==0], bins=bins, alpha=0.15, color=color, density=True)
    ax.hist(scores[y_all==1], bins=bins, alpha=0.4,  color=color, density=True,
            label=f"{name} (EER={eer_c*100:.1f}%)", histtype="step", lw=2, ls=ls)

ax.axvline(0, color="black", ls="--", lw=1.5, label="threshold = 0")
ax.set_xlabel("Calibrated score")
ax.set_title("Score distributions — both modalities overlaid")
ax.legend(fontsize=9)

plt.suptitle("Calibrated score space — input to fusion", fontsize=12)
plt.tight_layout()
plt.show()

corr = np.corrcoef(oof_audio, oof_image)[0, 1]
print(f"Pearson correlation between audio and image scores: {corr:.3f}")
print(f"Low correlation → complementary systems → fusion should help")

## 2. Fusion strategies

All three strategies produce a single scalar score per sample.

In [ ]:
# Strategy 1: Simple average
scores_avg = (oof_audio + oof_image) / 2

# Strategy 2: Grid search over weight w (audio weight)
weights = np.linspace(0, 1, 101)
grid_eers = []
for w in weights:
    s = w * oof_audio + (1 - w) * oof_image
    eer, _ = compute_eer(s[y_all==1], s[y_all==0])
    grid_eers.append(eer)

best_w   = weights[np.argmin(grid_eers)]
best_eer = min(grid_eers)
scores_grid = best_w * oof_audio + (1 - best_w) * oof_image
print(f"Grid search: best w={best_w:.2f} (audio weight), EER={best_eer*100:.2f}%")

# Strategy 3: Logistic regression
X_fusion = np.column_stack([oof_audio, oof_image])
fusion_clf = LogisticRegression(C=1.0, max_iter=1000,
                                class_weight="balanced", random_state=67)
fusion_clf.fit(X_fusion, y_all)
scores_logreg = fusion_clf.decision_function(X_fusion)

w_audio_lr = fusion_clf.coef_[0, 0]
w_image_lr = fusion_clf.coef_[0, 1]
print(f"LogReg fusion weights: audio={w_audio_lr:.3f}, image={w_image_lr:.3f}")
print(f"Effective ratio audio:image = {w_audio_lr/(w_audio_lr+w_image_lr):.2f}:{w_image_lr/(w_audio_lr+w_image_lr):.2f}")

## 3. Results — comparison table

In [ ]:
systems = {
    "Audio only":     oof_audio,
    "Image only":     oof_image,
    "Avg (w=0.5)":    scores_avg,
    f"Grid (w={best_w:.2f})": scores_grid,
    "LogReg fusion":  scores_logreg,
}

print(f"{'System':<20} {'EER [%]':>9} {'min-DCF':>9} {'Threshold':>11}")
print("-" * 55)
results = []
for name, scores in systems.items():
    eer, thr_eer = compute_eer(scores[y_all==1], scores[y_all==0])
    dcf, thr_dcf = compute_min_dcf(scores[y_all==1], scores[y_all==0])
    print(f"{name:<20} {eer*100:>9.2f} {dcf:>9.4f} {thr_dcf:>11.4f}")
    results.append({"name": name, "eer": eer, "min_dcf": dcf, "threshold": thr_dcf,
                    "scores": scores})

best = min(results, key=lambda x: x["eer"])
print("-" * 55)
print(f"Best: {best['name']}  EER={best['eer']*100:.2f}%  min-DCF={best['min_dcf']:.4f}")

## 4. Visualizations

In [ ]:
# Grid search curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(weights, [e*100 for e in grid_eers],
        color=COLORS["fusion"], lw=2)
ax.axvline(best_w, color=COLORS["green"], ls="--", lw=2,
           label=f"best w={best_w:.2f}  EER={best_eer*100:.2f}%")
ax.axvline(0.5, color=COLORS["gray"], ls=":", lw=1.5, label="w=0.5 (equal weight)")

eer_audio, _ = compute_eer(oof_audio[y_all==1], oof_audio[y_all==0])
eer_image, _ = compute_eer(oof_image[y_all==1], oof_image[y_all==0])
ax.axhline(eer_audio*100, color=COLORS["audio"], ls="--", lw=1, alpha=0.7,
           label=f"audio alone {eer_audio*100:.1f}%")
ax.axhline(eer_image*100, color=COLORS["image"], ls="--", lw=1, alpha=0.7,
           label=f"image alone {eer_image*100:.1f}%")

ax.set_xlabel("Audio weight w  (image weight = 1−w)")
ax.set_ylabel("EER [%]")
ax.set_title("Grid search: optimal audio/image weight")
ax.legend(fontsize=8)

# Score distribution for best fusion
ax = axes[1]
best_scores = best["scores"]
bins = np.linspace(best_scores.min(), best_scores.max(), 35)
ax.hist(best_scores[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
        label="non-target", density=True)
ax.hist(best_scores[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
        label="target", density=True)
ax.axvline(best["threshold"], color=COLORS["green"], ls="--", lw=2,
           label=f"thresh={best['threshold']:.3f}")
ax.axvline(0, color=COLORS["gray"], ls=":", lw=1.5, label="ideal=0")
ax.set_xlabel("Fusion score")
ax.set_title(f"Score distribution — {best['name']}\nEER={best['eer']*100:.2f}%")
ax.legend(fontsize=8)

plt.suptitle("Fusion analysis", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# DET curves — all systems
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

SYSTEM_COLORS = {
    "Audio only":    COLORS["audio"],
    "Image only":    COLORS["image"],
    "Avg (w=0.5)":   COLORS["gray"],
    f"Grid (w={best_w:.2f})": "#16A085",
    "LogReg fusion": COLORS["fusion"],
}

fig, ax = plt.subplots(figsize=(7, 7))
for r in results:
    fpr, tpr, _ = roc_curve(y_all, r["scores"])
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    is_fusion = "fusion" in r["name"].lower() or "Grid" in r["name"] or "Avg" in r["name"]
    lw = 2.5 if is_fusion else 1.5
    ls = "-" if is_fusion else "--"
    color = SYSTEM_COLORS.get(r["name"], "black")
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=color, lw=lw, ls=ls,
            label=f"{r['name']}  EER={r['eer']*100:.1f}%")

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curves — all systems (OOF)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Decision boundary of LogReg fusion in 2D score space
fig, ax = plt.subplots(figsize=(7, 6))

# Background decision regions
a_min, a_max = oof_audio.min()-0.5, oof_audio.max()+0.5
i_min, i_max = oof_image.min()-0.5, oof_image.max()+0.5
aa, ii = np.meshgrid(np.linspace(a_min, a_max, 200),
                     np.linspace(i_min, i_max, 200))
grid_scores = fusion_clf.decision_function(
    np.column_stack([aa.ravel(), ii.ravel()])
).reshape(aa.shape)
ax.contourf(aa, ii, grid_scores, levels=[-100, 0, 100],
            colors=[COLORS["nontarget"], COLORS["target"]], alpha=0.08)
ax.contour(aa, ii, grid_scores, levels=[0],
           colors=["black"], linewidths=2)

# Samples
ax.scatter(oof_audio[y_all==0], oof_image[y_all==0],
           c=COLORS["nontarget"], alpha=0.4, s=20, label="non-target")
ax.scatter(oof_audio[y_all==1], oof_image[y_all==1],
           c=COLORS["target"], alpha=0.9, s=80, zorder=5, label="target")

ax.set_xlabel("Audio calibrated score")
ax.set_ylabel("Image calibrated score")
ax.set_title(f"LogReg fusion decision boundary\n"
             f"weights: audio={w_audio_lr:.2f}, image={w_image_lr:.2f}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Save fusion model

In [ ]:
fusion_data = {
    "fusion_clf":    fusion_clf,       # LogReg fusion model
    "best_w":        best_w,            # optimal audio weight from grid search
    "results":       [{k: v for k, v in r.items() if k != "scores"} for r in results],
}

with open(Path("../cache/fusion.pkl"), "wb") as f:
    pickle.dump(fusion_data, f)

print("Saved to cache/fusion.pkl")
print(f"\nFinal summary:")
print(f"  Audio alone:    EER={eer_audio*100:.2f}%")
print(f"  Image alone:    EER={eer_image*100:.2f}%")
best_r = min(results, key=lambda x: x['eer'])
print(f"  Best fusion:    EER={best_r['eer']*100:.2f}%  ({best_r['name']})")
print(f"  Fusion threshold: {best_r['threshold']:.4f}")